[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%202/ex5_attention_vs_parse.ipynb)

# ISBA 2411 · Natural Language Processing & AI
## Week 2 · Lecture 4 · Exercise 2: Attention vs the Parse
**Summer 2026 · ~15 minutes · work in pairs · no API key needed**

---

### Reading (preview, not required today)
Today is a teaser. We open the box in Week 5; this is just intuition. If you want a head start:

- **Tunstall** (Revised Edition) -- Ch. 3 *Transformer Anatomy* (attention, multi-head attention, the encoder).
- **J&M** (3rd-ed. draft, Jan 6 2026) -- Ch. 8 *Transformers* (self-attention from the ground up).
- **HOLLM** -- Ch. 3 *Looking Inside Transformer LLMs*.

One honest caveat to carry through the whole exercise: **attention is not a parse**. Some heads loosely track syntactic relations, many do not, and whether attention counts as an explanation at all is an open research debate.

### What you will do
- Pull the attention weights out of a small transformer (DistilBERT)
- Run it on the same ambiguous sentence you parsed in Exercise 1
- Plot one layer and head as a token-by-token heatmap
- Lay that heatmap next to the dependency parse and compare

### What to bring back to the room
- Does any head line up with the arcs you drew (subject, object, modifier)?
- On the ambiguous sentence, does attention commit to one reading the way the parser did?
- Is attention a parse, or just something that sometimes rhymes with one?

### Step 0. Install and load
DistilBERT is small and fast on the free Colab tier. About 30 seconds.

In [ ]:
!pip install -q transformers torch

import torch
from transformers import AutoTokenizer, AutoModel

name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name, output_attentions=True)
model.eval()
print("Loaded", name, "-", model.config.n_layers, "layers,", model.config.n_heads, "heads")

### Step 1. Run the model and grab the attention
`out.attentions` is a tuple with one tensor per layer, each shaped `[batch, heads, tokens, tokens]`. Entry `[i, j]` is how much token *i* attends to token *j*.

In [ ]:
sentence = "I saw the man with the telescope"

enc = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

with torch.no_grad():
    out = model(**enc)

attentions = out.attentions
print("layers:", len(attentions), " shape per layer:", tuple(attentions[0].shape))
print("tokens:", tokens)

### Step 2. Plot one head as a heatmap
Each row is a token attending; each column is a token being attended to. Brighter means more attention. Change `layer` and `head` and watch the pattern change completely.

In [ ]:
import matplotlib.pyplot as plt

layer, head = 3, 4   # <-- try other values, 0..5 and 0..11
A = attentions[layer][0, head].numpy()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(A, cmap="Blues")
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
ax.set_xlabel("attended to"); ax.set_ylabel("attending from")
ax.set_title(f"DistilBERT attention - layer {layer}, head {head}")
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

### Step 3. The dependency parse, side by side
Same sentence, the spaCy parse from Exercise 1. Put this picture next to your heatmap.

In [ ]:
!pip install -q spacy
import spacy
from spacy import displacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

doc = nlp(sentence)
print([(t.text, t.dep_, "->", t.head.text) for t in doc])
displacy.render(doc, style="dep", jupyter=True, options={"compact": True, "distance": 110})

### Step 4. Your turn
Swap in your own sentence, ideally a structurally ambiguous one, and compare the heatmap to your intuition.

In [ ]:
your_sentence = "The chicken is ready to eat"   # <-- change me

enc = tokenizer(your_sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
with torch.no_grad():
    out = model(**enc)
A = out.attentions[layer][0, head].numpy()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(A, cmap="Blues")
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
ax.set_title(f"your sentence - layer {layer}, head {head}")
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

### Debrief
1. Did any single head clearly track a grammatical relation (verb to subject, noun to modifier), or were most heads doing something that does not look like syntax at all?
2. On *I saw the man with the telescope*, did attention favor one reading (who has the telescope), or did it spread the weight?
3. The parser made one hard choice per relation. Attention spread soft weight over everything. Which is more honest about ambiguity, and which is more useful downstream?

**The takeaway for Week 5:** attention lets every token look at every other token with a learned weight, and stacking many of these is most of what a transformer is. It *rhymes* with the arcs you drew, but it is not a parse, and reading syntax into a heatmap is a trap worth naming. We build the real mechanism next.